# CUDA destekli OpenCV ve C++ ile toplu boyutlandırma

Bu defter OpenCV 4.14.0 C++ kitaplığını **CUDA açık** olarak derler. Ardından C++ programı bir dizindeki görüntüleri **1024×768** boyutunda kaydeder. CPU, çok iş parçacıklı CPU ve OpenCV CUDA çalıştırmaları bulunur.

**Başlamadan:** Colab menüsünde **Çalışma zamanı → Çalışma zamanı türünü değiştir → GPU** seçin. Hücreleri yukarıdan aşağıya çalıştırın. Kaynak derlemesi uzun sürebilir; Colab oturumunu açık tutun. Ücretsiz GPU erişimi garanti edilmez.

**Teslim durumu:** Bu dosya çalıştırılmamış bir defter olarak hazırlanmıştır; gerçek Colab/CUDA başarısı iddia etmez. Başarılı çalıştırmadan sonra çıktıları koruyarak defteri indirin. Yerel CPU C++ test raporu ayrı dosyadadır.

Bu çalışma C++ OpenCV SDK kurar; Colab'ın önceden kurulu Python `cv2` paketini CUDA'lı sürüm diye kullanmaz. Kaynak kodlar defterin içinde bulunduğundan başka dosya yüklemek gerekmez. Kod hücrelerinde yorum satırı yoktur.


In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = Path('/content/opencv_odevi')
ROOT.mkdir(parents=True, exist_ok=True)
subprocess.run(['nvidia-smi'], check=True)
subprocess.run(['nvcc', '--version'], check=True)


## 1. Kaynak dosyalarını oluştur
C++17, CMake ve kurulum/test betikleri bu hücreyle yazılır.

In [ ]:
SOURCES = {'cpp/CMakeLists.txt': 'cmake_minimum_required(VERSION 3.18)\nproject(goruntu_boyutlandir LANGUAGES CXX)\noption(ENABLE_CUDA "OpenCV CUDA resize destegi" OFF)\nset(CMAKE_CXX_STANDARD 17)\nset(CMAKE_CXX_STANDARD_REQUIRED ON)\nset(CMAKE_CXX_EXTENSIONS OFF)\nfind_package(Threads REQUIRED)\nif(ENABLE_CUDA)\n    find_package(OpenCV 4 REQUIRED COMPONENTS core imgproc imgcodecs cudawarping)\nelse()\n    find_package(OpenCV 4 REQUIRED COMPONENTS core imgproc imgcodecs)\nendif()\nadd_executable(boyutlandir resize_images.cpp)\ntarget_include_directories(boyutlandir PRIVATE ${OpenCV_INCLUDE_DIRS})\ntarget_link_libraries(boyutlandir PRIVATE ${OpenCV_LIBS} Threads::Threads)\nif(ENABLE_CUDA)\n    target_compile_definitions(boyutlandir PRIVATE ENABLE_CUDA=1)\n    add_executable(cuda_kontrol cuda_check.cpp)\n    target_include_directories(cuda_kontrol PRIVATE ${OpenCV_INCLUDE_DIRS})\n    target_link_libraries(cuda_kontrol PRIVATE ${OpenCV_LIBS})\nendif()\nif(MSVC)\n    target_compile_options(boyutlandir PRIVATE /W4 /utf-8)\nelse()\n    target_compile_options(boyutlandir PRIVATE -Wall -Wextra -Wpedantic)\nendif()\n', 'cpp/resize_images.cpp': '#include <opencv2/core.hpp>\n#include <opencv2/imgcodecs.hpp>\n#include <opencv2/imgproc.hpp>\n#ifdef ENABLE_CUDA\n#include <opencv2/core/cuda.hpp>\n#include <opencv2/cudawarping.hpp>\n#endif\n#include <algorithm>\n#include <atomic>\n#include <cctype>\n#include <filesystem>\n#include <fstream>\n#include <iostream>\n#include <iterator>\n#include <mutex>\n#include <set>\n#include <stdexcept>\n#include <string>\n#include <thread>\n#include <vector>\n\nnamespace fs = std::filesystem;\n\ncv::Mat read_image(const fs::path& file) {\n    std::ifstream input(file, std::ios::binary);\n    if (!input) return {};\n    std::vector<unsigned char> bytes((std::istreambuf_iterator<char>(input)), {});\n    if (bytes.empty()) return {};\n    return cv::imdecode(bytes, cv::IMREAD_UNCHANGED);\n}\n\nvoid write_png(const fs::path& file, const cv::Mat& image) {\n    std::vector<unsigned char> bytes;\n    if (!cv::imencode(".png", image, bytes))\n        throw std::runtime_error("PNG kodlanamadi");\n    std::ofstream output(file, std::ios::binary);\n    if (!output) throw std::runtime_error("Cikti dosyasi acilamadi");\n    output.write(reinterpret_cast<const char*>(bytes.data()),\n                 static_cast<std::streamsize>(bytes.size()));\n    output.close();\n    if (!output) throw std::runtime_error("Cikti yazilamadi");\n}\n\nint main(int argc, char** argv) {\n    try {\n        if (argc < 3) {\n            std::cerr << "Kullanim: boyutlandir GIRDI_DIZINI CIKTI_DIZINI [--cuda] [--threads N]\\n";\n            return 1;\n        }\n        bool use_cuda = false;\n        int threads = 1;\n        for (int i = 3; i < argc; ++i) {\n            const std::string arg = argv[i];\n            if (arg == "--cuda") use_cuda = true;\n            else if (arg == "--threads" && i + 1 < argc) {\n                const std::string value = argv[++i];\n                std::size_t used = 0;\n                threads = std::stoi(value, &used);\n                if (used != value.size() || threads < 1 || threads > 64)\n                    throw std::runtime_error("Is parcacigi sayisi 1..64 olmali");\n            } else throw std::runtime_error("Bilinmeyen veya eksik arguman: " + arg);\n        }\n        const fs::path input = fs::u8path(argv[1]);\n        const fs::path output = fs::u8path(argv[2]);\n        if (!fs::is_directory(input)) throw std::runtime_error("Girdi dizini bulunamadi");\n        if (fs::weakly_canonical(input) == fs::weakly_canonical(output))\n            throw std::runtime_error("Girdi ve cikti dizinleri farkli olmali");\n        if (use_cuda) {\n#ifdef ENABLE_CUDA\n            if (cv::cuda::getCudaEnabledDeviceCount() < 1)\n                throw std::runtime_error("Kullanilabilir CUDA GPU yok");\n            if (!cv::cuda::DeviceInfo(0).isCompatible())\n                throw std::runtime_error("OpenCV derlemesi GPU mimarisi ile uyumsuz");\n            cv::cuda::setDevice(0);\n            cv::cuda::printShortCudaDeviceInfo(0);\n#else\n            throw std::runtime_error("Program CUDA destegiyle derlenmemis");\n#endif\n        }\n        const std::set<std::string> extensions = {\n            ".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp", ".pgm", ".ppm"\n        };\n        std::vector<fs::path> files;\n        for (const auto& entry : fs::directory_iterator(input)) {\n            if (!entry.is_regular_file()) continue;\n            std::string ext = entry.path().extension().string();\n            std::transform(ext.begin(), ext.end(), ext.begin(),\n                           [](unsigned char c) { return static_cast<char>(std::tolower(c)); });\n            if (extensions.count(ext)) files.push_back(entry.path());\n        }\n        std::sort(files.begin(), files.end());\n        if (files.empty()) throw std::runtime_error("Dizinde desteklenen goruntu dosyasi yok");\n        fs::create_directories(output);\n        cv::setNumThreads(1);\n        std::atomic<std::size_t> next{0}, done{0}, skipped{0}, failed{0};\n        std::mutex log_mutex;\n        auto worker = [&]() {\n            while (true) {\n                const auto index = next.fetch_add(1);\n                if (index >= files.size()) break;\n                const auto& file = files[index];\n                try {\n                    const cv::Mat source = read_image(file);\n                    if (source.empty()) {\n                        ++skipped;\n                        std::lock_guard<std::mutex> lock(log_mutex);\n                        std::cerr << "ATLANDI: " << file.filename().u8string() << \'\\n\';\n                        continue;\n                    }\n                    if ((source.depth() != CV_8U && source.depth() != CV_16U) ||\n                        (source.channels() != 1 && source.channels() != 3 && source.channels() != 4))\n                        throw std::runtime_error("PNG icin 8/16 bit ve 1/3/4 kanal gerekli");\n                    const fs::path target = output / fs::u8path(file.filename().u8string() + ".png");\n                    if (fs::exists(target)) throw std::runtime_error("Cikti zaten var; farkli dizin secin");\n                    cv::Mat resized;\n                    if (use_cuda) {\n#ifdef ENABLE_CUDA\n                        cv::cuda::setDevice(0);\n                        cv::cuda::GpuMat src_gpu, dst_gpu;\n                        src_gpu.upload(source);\n                        cv::cuda::resize(src_gpu, dst_gpu, cv::Size(1024, 768), 0, 0, cv::INTER_LINEAR);\n                        dst_gpu.download(resized);\n#endif\n                    } else {\n                        cv::resize(source, resized, cv::Size(1024, 768), 0, 0, cv::INTER_LINEAR);\n                    }\n                    write_png(target, resized);\n                    ++done;\n                    std::lock_guard<std::mutex> lock(log_mutex);\n                    std::cout << file.filename().u8string() << " : " << source.cols << \'x\'\n                              << source.rows << " -> 1024x768 : " << target.filename().u8string() << \'\\n\';\n                } catch (const std::exception& e) {\n                    ++failed;\n                    std::lock_guard<std::mutex> lock(log_mutex);\n                    std::cerr << "HATA: " << file.filename().u8string() << " : " << e.what() << \'\\n\';\n                }\n            }\n        };\n        std::vector<std::thread> pool;\n        const auto count = std::min<std::size_t>(static_cast<std::size_t>(threads), files.size());\n        try {\n            for (std::size_t i = 0; i < count; ++i) pool.emplace_back(worker);\n        } catch (...) {\n            for (auto& thread : pool) thread.join();\n            throw;\n        }\n        for (auto& thread : pool) thread.join();\n        std::cout << "TAMAM: " << done << " ATLANDI: " << skipped << " HATA: " << failed << \'\\n\';\n        return failed ? 3 : (done ? 0 : 2);\n    } catch (const std::exception& e) {\n        std::cerr << "HATA: " << e.what() << \'\\n\';\n        return 1;\n    }\n}\n', 'cpp/cuda_check.cpp': '#include <opencv2/core.hpp>\n#include <opencv2/core/cuda.hpp>\n#include <opencv2/cudawarping.hpp>\n#include <iostream>\n\nint main() {\n    try {\n        std::cout << cv::getBuildInformation() << \'\\n\';\n        const int count = cv::cuda::getCudaEnabledDeviceCount();\n        std::cout << "CUDA_DEVICE_COUNT=" << count << \'\\n\';\n        if (count < 1 || !cv::cuda::DeviceInfo(0).isCompatible()) return 1;\n        cv::cuda::setDevice(0);\n        cv::cuda::printShortCudaDeviceInfo(0);\n        cv::Mat source(16, 16, CV_8UC1, cv::Scalar(123));\n        cv::cuda::GpuMat input, output;\n        input.upload(source);\n        cv::cuda::resize(input, output, cv::Size(1024, 768), 0, 0, cv::INTER_LINEAR);\n        cv::Mat result;\n        output.download(result);\n        if (result.cols != 1024 || result.rows != 768 || cv::countNonZero(result != 123)) return 2;\n        std::cout << "CUDA_RESIZE_TEST=PASS\\n";\n        return 0;\n    } catch (const cv::Exception& e) {\n        std::cerr << e.what() << \'\\n\';\n        return 3;\n    }\n}\n', 'cpp/test_resize.py': "import argparse\nimport json\nfrom pathlib import Path\nimport subprocess\nimport tempfile\nimport cv2\nimport numpy as np\n\n\ndef save(path, a):\n    ok, data = cv2.imencode(path.suffix, a)\n    assert ok\n    data.tofile(str(path))\n\n\ndef read(path):\n    return cv2.imdecode(np.fromfile(str(path), np.uint8), cv2.IMREAD_UNCHANGED)\n\n\ndef test(exe, destination, cuda=False):\n    destination = Path(destination).resolve()\n    destination.mkdir(parents=True, exist_ok=True)\n    root = Path(tempfile.mkdtemp(prefix='cpp_test_', dir=destination))\n    inp = root / 'girdiler'\n    inp.mkdir()\n    rng = np.random.default_rng(42)\n    samples = {\n        'renk.png': rng.integers(0, 256, (123, 217, 3), dtype=np.uint8),\n        'gri.png': rng.integers(0, 256, (59, 83), dtype=np.uint8),\n        'derin.png': rng.integers(0, 65536, (91, 101), dtype=np.uint16),\n        'alpha.png': rng.integers(0, 256, (27, 53, 4), dtype=np.uint8),\n        'ayni.png': np.full((20, 30), 81, np.uint8),\n        'ayni.jpg': np.full((30, 20), 201, np.uint8),\n        'buyuk.PNG': np.full((900, 1400, 3), 123, np.uint8)\n    }\n    for name, image in samples.items():\n        save(inp / name, image)\n    (inp / 'bozuk.png').write_bytes(b'not an image')\n    (inp / 'metin.txt').write_text('image olmayan dosya', encoding='utf-8')\n    (inp / 'alt_dizin').mkdir()\n    save(inp / 'alt_dizin' / 'dahil_degil.png', np.zeros((2, 2), np.uint8))\n    commands = []\n\n    def run(args, expected=0):\n        p = subprocess.run([str(exe), *map(str, args)], capture_output=True, text=True, encoding='utf-8', errors='replace')\n        commands.append({'args': list(map(str, args)), 'exit_code': p.returncode,\n                         'stdout': p.stdout, 'stderr': p.stderr})\n        assert p.returncode == expected, p.stdout + p.stderr\n\n    for mode, args in [('serial', []), ('parallel', ['--threads', '4'])] + ([('cuda', ['--cuda'])] if cuda else []):\n        out = root / mode\n        run([inp, out, *args])\n        assert len(list(out.glob('*.png'))) == len(samples)\n        for name in samples:\n            source = read(inp / name)\n            result = read(out / (name + '.png'))\n            assert result is not None and result.shape[:2] == (768, 1024)\n            assert result.dtype == source.dtype and result.ndim == source.ndim\n            if mode != 'cuda':\n                ref = cv2.resize(source, (1024, 768), interpolation=cv2.INTER_LINEAR)\n                assert np.abs(result.astype(float) - ref.astype(float)).max() <= 1\n            if mode == 'parallel':\n                assert np.array_equal(result, read(root / 'serial' / (name + '.png')))\n        if mode == 'cuda':\n            assert np.all(read(out / 'buyuk.PNG.png') == 123)\n    run([inp, root / 'serial'], 3)\n    run([inp, inp], 1)\n    run([root / 'olmayan', root / 'yeni'], 1)\n    run([inp, root / 'bad_threads', '--threads', '0'], 1)\n    empty = root / 'bos'\n    empty.mkdir()\n    run([empty, root / 'bos_cikti'], 1)\n    if not cuda:\n        run([inp, root / 'cuda_yok', '--cuda'], 1)\n    report = {'status': 'PASS', 'cuda_tested': cuda, 'images_per_mode': len(samples),\n              'dimensions': [1024, 768], 'commands': commands}\n    (destination / 'cpp_test_raporu.json').write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding='utf-8')\n    print(json.dumps({k: v for k, v in report.items() if k != 'commands'}, indent=2))\n    return root\n\n\nif __name__ == '__main__':\n    p = argparse.ArgumentParser()\n    p.add_argument('executable', type=Path)\n    p.add_argument('--out', type=Path, default=Path('cpp_test_sonuclari'))\n    p.add_argument('--cuda', action='store_true')\n    args = p.parse_args()\n    test(args.executable.resolve(), args.out, args.cuda)\n", 'colab_setup.py': 'import os\nfrom pathlib import Path\nimport re\nimport shutil\nimport subprocess\nimport tarfile\nimport urllib.request\n\n\ndef run(args, log):\n    print(\' \'.join(map(str, args)), flush=True)\n    with log.open(\'w\', encoding=\'utf-8\') as stream:\n        result = subprocess.run(list(map(str, args)), stdout=stream, stderr=subprocess.STDOUT)\n    text = log.read_text(encoding=\'utf-8\', errors=\'replace\')\n    print(text[-5000:], flush=True)\n    if result.returncode:\n        raise RuntimeError(f\'Komut basarisiz. Tam gunluk: {log}\')\n    return text\n\n\ndef install(root=Path(\'/content/opencv_odevi\')):\n    root = Path(root).resolve()\n    root.mkdir(parents=True, exist_ok=True)\n    if not shutil.which(\'nvidia-smi\') or not shutil.which(\'nvcc\'):\n        raise RuntimeError(\'Colab GPU oturumunu secin. nvidia-smi ve nvcc bulunmali.\')\n    run([\'nvidia-smi\'], root / \'nvidia-smi.txt\')\n    run([\'nvcc\', \'--version\'], root / \'nvcc.txt\')\n    archs = subprocess.check_output(\n        [\'nvidia-smi\', \'--query-gpu=compute_cap\', \'--format=csv,noheader\'], text=True).splitlines()\n    arch = archs[0].strip() if archs else \'\'\n    if not re.fullmatch(r\'\\d+\\.\\d+\', arch):\n        raise RuntimeError(\'GPU compute capability okunamadi; CUDA_ARCH_BIN otomatik secilemedi.\')\n    run([\'apt-get\', \'update\', \'-qq\'], root / \'apt-update.log\')\n    run([\'apt-get\', \'install\', \'-y\', \'-qq\', \'build-essential\', \'cmake\', \'ninja-build\',\n         \'libjpeg-dev\', \'libpng-dev\', \'libtiff-dev\', \'libwebp-dev\'], root / \'apt-install.log\')\n    version = \'4.14.0\'\n    for name in (\'opencv\', \'opencv_contrib\'):\n        source = root / f\'{name}-{version}\'\n        if not source.is_dir():\n            archive = root / f\'{name}-{version}.tar.gz\'\n            urllib.request.urlretrieve(f\'https://github.com/opencv/{name}/archive/refs/tags/{version}.tar.gz\', archive)\n            with tarfile.open(archive) as tar:\n                tar.extractall(root, filter=\'data\')\n    prefix = root / \'install\'\n    build = root / \'opencv-build\'\n    config = run([\n        \'cmake\', \'-S\', root / f\'opencv-{version}\', \'-B\', build, \'-G\', \'Ninja\',\n        \'-DCMAKE_BUILD_TYPE=Release\', f\'-DCMAKE_INSTALL_PREFIX={prefix}\',\n        f\'-DOPENCV_EXTRA_MODULES_PATH={root / f"opencv_contrib-{version}" / "modules"}\',\n        \'-DWITH_CUDA=ON\', f\'-DCUDA_ARCH_BIN={arch}\', \'-DCUDA_ARCH_PTX=\',\n        \'-DBUILD_LIST=core,imgproc,imgcodecs,cudawarping\', \'-DBUILD_SHARED_LIBS=ON\',\n        \'-DBUILD_TESTS=OFF\', \'-DBUILD_PERF_TESTS=OFF\', \'-DBUILD_EXAMPLES=OFF\',\n        \'-DBUILD_opencv_python2=OFF\', \'-DBUILD_opencv_python3=OFF\', \'-DBUILD_JAVA=OFF\',\n        \'-DWITH_CUDNN=OFF\', \'-DOPENCV_DNN_CUDA=OFF\', \'-DWITH_CUBLAS=OFF\',\n        \'-DWITH_CUFFT=OFF\', \'-DWITH_NVCUVID=OFF\', \'-DWITH_NVCUVENC=OFF\',\n        \'-DWITH_IPP=OFF\', \'-DWITH_OPENCL=OFF\', \'-DWITH_FFMPEG=OFF\',\n        \'-DWITH_GSTREAMER=OFF\', \'-DWITH_GTK=OFF\', \'-DWITH_QT=OFF\',\n        \'-DOPENCV_GENERATE_PKGCONFIG=ON\'\n    ], root / \'opencv-configure.log\')\n    if not re.search(r\'NVIDIA CUDA:\\s+YES\', config):\n        raise RuntimeError(\'CMake NVIDIA CUDA: YES raporlamadi. CPU derlemesine devam edilmiyor.\')\n    jobs = str(min(2, os.cpu_count() or 1))\n    run([\'cmake\', \'--build\', build, \'--parallel\', jobs], root / \'opencv-build.log\')\n    run([\'cmake\', \'--install\', build], root / \'opencv-install.log\')\n    print(f\'OpenCV C++ CUDA kurulumu tamamlandi: {prefix}\', flush=True)\n    return prefix\n\n\nif __name__ == \'__main__\':\n    install()\n'}
for name, text in SOURCES.items():
    path = ROOT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding='utf-8')
print('Kaynak dosyalari hazir.')


## 2. CUDA destekli OpenCV kurulumu
GPU mimarisi otomatik belirlenir. OpenCV ve opencv_contrib aynı 4.14.0 sürümünden indirilir. CMake çıktısında `NVIDIA CUDA: YES` yoksa işlem durur. Bellek kullanımını sınırlamak için en fazla iki derleme işi kullanılır. Ayrıntılı günlükler `/content/opencv_odevi` içindedir.


In [ ]:
subprocess.run([sys.executable, '-u', str(ROOT / 'colab_setup.py')], check=True)


## 3. C++ programını derle ve CUDA kurulumunu doğrula
Yalnız cihaz sayısı değil, gerçek bir `cv::cuda::resize` işlemi de test edilir. Son satır `CUDA_RESIZE_TEST=PASS` olmalıdır.

In [ ]:
BUILD = ROOT / 'cpp-build'
subprocess.run(['cmake', '-S', str(ROOT / 'cpp'), '-B', str(BUILD), '-G', 'Ninja',
                '-DCMAKE_BUILD_TYPE=Release', '-DENABLE_CUDA=ON',
                f'-DOpenCV_DIR={ROOT / "install/lib/cmake/opencv4"}'], check=True)
subprocess.run(['cmake', '--build', str(BUILD), '--parallel', '2'], check=True)
verification = subprocess.run([str(BUILD / 'cuda_kontrol')], capture_output=True, text=True)
(ROOT / 'cuda_dogrulama.txt').write_text(verification.stdout + verification.stderr, encoding='utf-8')
print(verification.stdout)
print(verification.stderr)
verification.check_returncode()
assert 'CUDA_RESIZE_TEST=PASS' in verification.stdout


## 4. Girdi dizinini hazırla
Dört sentetik örnek eklenir. Kendi görüntülerinizi de kullanmak için aşağıdaki değişkeni `True` yapın. Program dizinin doğrudan içindeki desteklenen görüntüleri okur; alt dizinlere inmez.

In [ ]:
import base64
import datetime

RUN = ROOT / ('calisma_' + datetime.datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
INPUT = RUN / 'girdiler'
INPUT.mkdir(parents=True, exist_ok=True)
EXAMPLES = {'dusuk_kontrast.png': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAAAAACfHP67AAAel0lEQVR4AcTBW3Id53og0cwBWCJCFHkk1Phq5LxI3OSbj4jHLqAI/F86OqLbtlRYy52SQUJCBgkpWaQkZJCQkEFKQgYJCRkkpGSRkpBBTu6UDBISMkhIySIlIYOEhAxSEjJISMggISWLlIQMcnKnZJCQkEFCShYpCRkkJGSQkpBBQkIGCSlZpCRkkJM7JYOEhAwSUrJIScggISGDlIQMEhIySEjJIiUhg5zcKRkkJGSQkJJFSkIGCQkZpCRkkJCQQUJKFikJGeTkTskgISGDhJQsUhIySEjIICUhg4SEDBJSskhJyCAnd0oGCQkZJKRkkZKQQUJCBikJGSQkZJCQkkVKQgY5uVMySEjIICEli5SEDBISMkhJyCAhIYOElCxSEjLIyZ2SQUJCBgkpWaQkZJCQkEFKQgYJCRkkpGSRkpBBTu6UDBISMkhIySIlIYOEhAxSEjJISMggISWLlIQMcnKnZJCQkEFCShYpCRkkJGSQkpBBQkIGCSlZpCRkkJM7JYOEhAwSUrJIScggISGDlIQMEhIySEjJIiUhg5zcKRkkJGSQkJJFSkIGCQkZpCRkkJCQQUJKFikJGeTkTskgISGDhJQsUhIySEjIICUhg4SEDBJSskhJyCAnd0oGCQkZJKRkkZKQQUJCBikJGSQkZJCQkkVKQgY5uVMySEjIICEli5SEDBISMkhJyCAhIYOElCxSEjLIyZ2SQUJCBgkpWaQkZJCQkEFKQgYJCRkkpGSRkpBBTu6UDBISMkhIySIlIYOEhAxSEjJISMggISWLlIQMcnKnZJCQkEFCShYpCRkkJGSQkpBBQkIGCSlZpCRkkJM7JYOEhAwSUrJIScggISGDlIQMEhIySEjJIiUhg5zcKRkkJGSQkJJFSkIGCQkZpCRkkJCQQUJKFikJGeTkTskgISGDhJQsUhIySEjIICUhg4SEDBJSskhJyCAnd0oGCQkZJKRkkZKQQUJCBikJGSQkZJCQkkVKQgY5uVMySEjIICEli5SEDBISMkhJyCAhIYOElCxSEjLIyZ2SQUJCBgkpWaQkZJCQkEFKQgYJCRkkpGSRkpBBTu6UDBISMkhIySIlIYOEhAxSEjJISMggISWLlIQMcnKnZJCQkEFCShYpCRkkJGSQkpBBQkIGCSlZpCRkkJM7JYOEhAwSUrJIScggISGDlIQMEhIySEjJIiUhg5zcKRkkJGSQkJJFSkIGCQkZpCRkkJCQQUJKFikJGeTkTskgISGDhJQsUhIySEjIICUhg4SEDBJSskhJyCAnd0oGNy514yCDhIQMUhIySEjIICEli5SEDHJyp2Rw41I3DjJISMggJSGDhIQMElKySEnIICd3SgY3LnXjIIOEhAxSEjJISMggISWLlIQMcnKnZHDjUjcOMkhIyCAlIYOEhAwSUrJIScggJ3dKBjcudeMgg4SEDFISMkhIyCAhJYuUhAxycqdkcONSNw4ySEjIICUhg4SEDBJSskhJyCAnd0oGNy514yCDhIQMUhIySEjIICEli5SEDHJyp2Rw41I3DjJISMggJSGDhIQMElKySEnIICd3SgY3LnXjIIOEhAxSEjJISMggISWLlIQMcnKnZHDjUjcOMkhIyCAlIYOEhAwSUrJIScggJ3dKBjcudeMgg4SEDFISMkhIyCAhJYuUhAxycqdkcONSNw4ySEjIICUhg4SEDBJSskhJyCAnd0oGNy514yCDhIQMUhIySEjIICEli5SEDHJyp2Rw41I3DjJISMggJSGDhIQMElKySEnIICd3SgY3LnXjIIOEhAxSEjJISMggISWLlIQMcnKnZHDjUjcOMkhIyCAlIYOEhAwSUrJIScggJ3dKBjcudeMgg4SEDFISMkhIyCAhJYuUhAxycqdkcONSNw4ySEjIICUhg4SEDBJSskhJyCAnd0oGNy514yCDhIQMUhIySEjIICEli5SEDHJyp2Rw41I3DjJISMggJSGDhIQMElKySEnIICd3SgY3LnXjIIOEhAxSEjJISMggISWLlIQMcnKnZHDjUjcOMkhIyCAlIYOEhAwSUrJIScggJ3dKBjcudeMgg4SEDFISMkhIyCAhJYuUhAxycqdkcONSNw4ySEjIICUhg4SEDBJSskhJyCAnd0oGNy514yCDhIQMUhIySEjIICEli5SEDHJyp2Rw41I3DjJISMggJSHLf/4HISGDhJQsUhIyyMmdksGNS904yCAhIYOUhP/J/9V/ABIySEjJIiUhg5zcKRncuNSNgwwSEjJIyYt/8//wE5MMElKySEnIICd3SgY3LnXjIIOEhAxScvo3/19+4oUMElKySEnIICd3SgY3LnXjIIOEhAxScvg3/w0/cZJBQkoWKQkZ5OROyeDGpW4cZJCQkEFK/s1/208cZJCQkkVKQgY5uVMyuHGpGwcZJCRkkPqL/5GfQAYJKVmkJGSQkzslgxuXunGQQUJCBpn+4n/sJxkkpGSRkpBBTu6UDG5c6sZBBgkJGWT5i/+Vn1kkpGSRkpBBTu6UDG5c6sZBBgkJGeTZX/yv/cwzCSlZpCRkkJM7JYMbl7pxkEFCQgb54S/+Bj/zg4SULFISMsjJnZLBjUvdOMggISGDPPmLv8nPPJGQkkVKQgY5uVMyuHGpGwcZJCRkkEd/8bf5mUcSUrJIScggJ3dKBjcudeMgg4SEDAJ/8bf6GZCQkkVKQgY5uVMyuHGpGwcZJCRkEP7ib/YzSEjJIiUhg5zcKRncuNSNgwwSEjLIX/ztfkZCShYpCRnk5E7J4MalbhxkkJCQwb/4B/wsISWLlIQMcnKnZHDjUjcOMkhIyPCdf8QbQkoWKQkZ5OROyeDGpW4cZJCQkOU7/5A3TFKySEnIICd3SgY3LnXjIIOEhLz4zj/mDYOULFISMsjJnZLBjUvdOMggISHPvvMPesMiJYuUhAxycqdkcONSNw4ySEjID9/5R73hhZQsUhIyyMmdksGNS904yCAhIafv/MPe8ExKFikJGeTkTsngxqVuHGSQkJAn3/nHveEHKVmkJGSQkzslgxuXunGQQUJCHn3nAm84SckiJSGDnNwpGdy41I2DDBIScvjOJd7wREoWKQkZ5OROyeDGpW4cZJCQEPjORd7wSEoWKQkZ5OROyeDGpW4cZJCQEPjORd7wSEoWKQkZ5OROyeDGpW4cZJCQEL5zmTccpGSRkpBBTu6UDG5c6sZBBgkJ+c6F3gBSskhJyCAnd0oGNy514yCDhIR850JvAClZpCRkkJM7JYMbl7pxkEFCwu9c6g1IySIlIYOc3CkZ3LjUjYMMEhIPXOwNUrJIScggJ3dKBjcudeMgg4TEAxd7g5QsUhIyyMmdksGNS904yCAh0wOXeyMli5SEDHJyp2Rw41I3DjJIyPTA5d5IySIlIYOc3CkZ3LjUjYMMEjI88AruKFmkJGSQkzslgxuXunGQQUKGB17BHSWLlIQMcnKnZHDjUjcOMkjI8sCruCNkkZKQQU7ulAxuXOrGQQYJWR54FXeELFISMsjJnZLBjUvdOMggIS8eeCV3TLJIScggJ3dKBjcudeMgg4S8eOCV3DHJIiUhg5zcKRncuNSNgwwS8uKBV3LHJIuUhAxycqdkcONSNw4ySMizB17NHYMsUhIyyMmdksGNS904yCAhzx54NXcMskhJyCAnd0oGNy514yCDhDx74NXcMcgiJSGDnNwpGdy41I2DDBLywwOv6I5FFikJGeTkTsngxqVuHGSQkB8eeEV3LLJIScggJ3dKBjcudeMgg4T88MArumORRUpCBjm5UzK4cakbBxkk5PTAq7rjhSxSEjLIyZ2SwY1L3TjIICGnB17VHS9kkZKQQU7ulAxuXOrGQQYJOT3wqu54IYuUhAxycqdkcONSNw4ySMjpgVd1xwtZpCRkkJM7JYMbl7pxkEFCTg+8qjteyCIlIYOc3CkZ3LjUjYMMEvLkgVd2xzNZpCRkkJM7JYMbl7pxkEFCnjzwyu54JouUhAxycqdkcONSNw4ySMiTB17ZHc9kkZKQQU7ulAxuXOrGQQYJefLAK7vjmSxSEjLIyZ2SwY1L3TjIICFPHnhldzyTRUpCBjm5UzK4cakbBxkk5MkDr+yOZ7JIScggJ3dKBjcudeMgg4Q8eeCV3fFMFikJGeTkTsngxqVuHGSQkCcPvLI7nskiJSGDnNwpGdy41I2DDBLy5IFXdsczWaQkZJCTOyWDG5e6cZBBQp488MrueCaLlIQMcnKnZHDjUjcOMkjIkwde2R3PZJGSkEFO7pQMblzqxkEGCXn0wKu74wdZpCRkkJM7JYMbl7pxkEFCnjzwyu54JouUhAxycqdkcONSNw4ySMiTB17ZHc9kkZKQQU7ulAxuXOrGQQYJefLAK7vjmSxSEjLIyZ2SwY1L3TjIICFPHnhldzyTRUpCBjm5UzK4cakbBxkk5MkDr+yOZ7JIScggJ3dKBjcudeMgg4Q8eeCV3fFMFikJGeTkTsngxqVuHGSQkCcPvLI7nskiJSGDnNwpGdy41I2DDBLy5IFXdsczWaQkZJCTOyWDG5e6cZBBQp488MrueCaLlIQMcnKnZHDjUjcOMkjIkwde2R3PZJGSkEFO7pQMblzqxkEGCXnywCu745ksUhIyyMmdksGNS904yCAhpwde1R0vZJGSkEFO7pQMblzqxkEGCTk98KrueCGLlIQMcnKnZHDjUjcOMkjI6YFXdccLWaQkZJCTOyWDG5e6cZBBQk4PvKo7XsgiJSGDnNwpGdy41I2DDBJyeuBV3fFCFikJGeTkTsngxqVuHGSQkB8eeEV3LLJIScggJ3dKBjcudeMgg4T88MArumORRUpCBjm5UzK4cakbBxkk5IcHXtEdiyxSEjLIyZ2SwY1L3TjIICHPHng1dwyySEnIICd3SgY3LnXjIIOEPHvg1dwxyCIlIYOc3CkZ3LjUjYMMEvLsgVdzxyCLlIQMcnKnZHDjUjcOMkjIiwdeyR2TLFISMsjJnZLBjUvdOMggIS8eeCV3TLJIScggJ3dKBjcudeMgg4S8eOCV3DHJIiUhg5zcKRncuNSNgwwSsjzwKu4IWaQkZJCTOyWDG5e6cZBBQpYHXsUdIYuUhAxycqdkcONSNw4ySMjwwCu4o2SRkpBBTu6UDG5c6sZBBgkZHngFd5QsUhIyyMmdksGNS904yCAh0wOXeyMli5SEDHJyp2Rw41I3DjJIyPTA5d5IySIlIYOc3CkZ3LjUjYMMEhIPXOwNUrJIScggJ3dKBjcudeMgg4TEAxd7g5QsUhIyyMmdksGNS904yCAh4Xcu9QakZJGSkEFO7pQMblzqxkEGCQn5zoXeAFKySEnIICd3SgY3LnXjIIOEhHznQm8AKVmkJGSQkzslgxuXunGQQUJC+M5l3nCQkkVKQgY5uVMyuHGpGwcZJCQEvnORNzySkkVKQgY5uVMyuHGpGwcZJCQEvnORNzySkkVKQgY5uVMyuHGpGwcZJCTk8J1LvOGJlCxSEjLIyZ2SwY1L3TjIICEhj75zgTecpGSRkpBBTu6UDG5c6sZBBgkJefKdf9wbfpCSRUpCBjm5UzK4cakbBxkkJOT0nX/YG55JySIlIYOc3CkZ3LjUjYMMEhLyw3f+UW94ISWLlIQMcnKnZHDjUjcOMkhIyLPv/IPesEjJIiUhg5zcKRncuNSNgwwSEvLiO/+YNwxSskhJyCAnd0oGNy514yCDhIQs3/mHvGGSkkVKQgY5uVMyuHGpGwcZJCRk+M4/4g0hJYuUhAxycqdkcONSNw4ySEjI4F/8A36WkJJFSkIGOblTMrhxqRsHGSQkZJC/+Nv9jISULFISMsjJnZJBQkIGCSlZpCRkkJCQQfiLv9nPICEli5SEDHJyp2SQkJBBQkoWKQkZJCRkEPiLv9XPgISULFISMsjJnZJBQkIGCSlZpCRkkJCQQR79xd/mZx5JSMkiJSGDnNwpGSQkZJCQkkVKQgYJCRnkyV/8TX7miYSULFISMsjJnZJBQkIGCSlZpCRkkJCQQX74i7/Bz/wgISWLlIQMcnKnZJCQkEFCShYpCRkkJGSQZ3/xv/YzzySkZJGSkEFO7pQMEhIySEjJIiUhg4SEDLL8xf/KzywSUrJIScggJ3dKBgkJGSSkZJGSkEFCQgaZ/uJ/7CcZJKRkkZKQQU7ulAwSEjJISMkiJSGDhIQMUn/xP/ITyCAhJYuUhAxycqdkkJCQQUJKFikJGSQkZJCSf/Pf9hMHGSSkZJGSkEFO7pQMEhIySEjJIiUhg4SEDFJy+Df/DT9xkkFCShYpCRnk5E7JICEhg4SULFISMkhIyCAlp3/z/+UnXsggISWLlIQMcnKnZJCQkEFCShYpCRkkJGSQkhf/5v/hJyYZJKRkkZKQQU7ulAwSEjJISMkiJSGDhIQMUhL+J/9X/wFIyCAhJYuUhAxycqdkkJCQQUJKFikJGSQkZJCSkOU//4OQkEFCShYpCRnk5E7JICEhg4SULFISMkhIyCAlIYOEhAwSUrJIScggJ3dKBgkJGSSkZJGSkEFCQgYpCRkkJGSQkJJFSkIGOblTMkhIyCAhJYuUhAwSEjJIScggISGDhJQsUhIyyMmdkkFCQgYJKVmkJGSQkJBBSkIGCQkZJKRkkZKQQU7ulAwSEjJISMkiJSGDhIQMUhIySEjIICEli5SEDHJyp2SQkJBBQkoWKQkZJCRkkJKQQUJCBgkpWaQkZJCTOyWDhIQMElKySEnIICEhg5SEDBISMkhIySIlIYOc3CkZJCRkkJCSRUpCBgkJGaQkZJCQkEFCShYpCRnk5E7JICEhg4SULFISMkhIyCAlIYOEhAwSUrJIScggJ3dKBgkJGSSkZJGSkEFCQgYpCRkkJGSQkJJFSkIGOblTMkhIyCAhJYuUhAwSEjJIScggISGDhJQsUhIyyMmdkkFCQgYJKVmkJGSQkJBBSkIGCQkZJKRkkZKQQU7ulAwSEjJISMkiJSGDhIQMUhIySEjIICEli5SEDHJyp2SQkJBBQk4bp48sUhIySEi843TjILzlKy/k8AvfOMkgISGDhJQsUhIyyMmdkkFCQgYJebSxfOSZlIQMErK8Y7kB8ha+8kIOv8A3nsggISGDhJQsUhIyyMmdkkFCQgYJOWxMH/lBSkIGCXnxjukG8hb4yjM5/ALfeCKDhIQMElKySEnIICd3SgYJCRkkBDbgA082Dh85SUnIICHP3nG48eRX4IbwFr7yTOAX+MZJBgkJGSSkZJGSkEFO7pQMEhIySAgb8IEfNuAjJykJGSTk2Tvgi5x+BW7CW/jKM4Ff4BsnGSQkZJCQkkVKQgY5uVMySEjIICFs8IFn3gMfeSIlIYOE/PAO+IL88CvcBN7CV34QfgG+cZJBQkIGCSlZpCRkkJM7JYOEhAwSssEHXsg98JFHUhIySMjpHfAF5Nmv8BV4C1/5QfgFvvGDDBISMkhIySIlIYOc3CkZJCRkkJANPvBCuIePPJKSkEFCTu/gCyDPfoWvwFvgKyfhF/jGDzJISMggISWLlIQMcnKnZJCQkEHCDT6wCPfwkUf+/gn4ncMnDrL8xqM/eCbv/wTe8+hP5PQOvgDy4tebHN7CV07yC3zjmQwSEjJISMkiJSGDnNwpGSQkZJBwgw8swj3wkcM9fPqdHz6BPPuNZ39w8j38+Z5nX3jyDvgCyCCHt/CVk/wC33gmg4SEDBJSskhJyCAnd0oGCQkZJNzgA4vAPXzkcM/0CfnhN5Y/eOJ74guP3sEXDjLI4S3wlSf+AnzjmQwSEjJISMkiJSGDnNwpGSQkZJBwgw8sAvfwkcM9jz4Bv3P4JKffOHxG/sXhDx75nid/Au+BLzx6B184yCCP3sJXnvgLfOOFDBISMkhIySIlIYOc3CkZJCRkkHCDDywC9/CRwz2HTzz5HfjMk9+Az4DAv+APHvmew588eQ9fePQOvnCQQR69Bb7yyF/gGy9kkJCQQUJKFikJGeTkTskgISGDhBt8YBG4h48c7oFP/PA78JlHvwGfATn8C/7g4HvgT07vgS8c3sEXDjLIk7fwlUdvgW+8kEFCQgYJKVmkJGSQkzslg4SEDBJu8IFF4B4+criHT7z4HT5z+A34zEEO/4I/OPge/uTZe/jC4R184SCDPHkLX3n0Fr6xyCAhIYOElCxSEjLIyZ2SQUJCBgk3+MAicA8fOdzDJ178Dp85/MZ/8QcH38OfPHsPXzi8gy8cZJAnb4GvHN7CNxYZJCRkkJCSRUpCBjm5UzJISMgg4QYfWATu4SOHe/jEi9+Bz8Bv/Bd/cPA9/Mmz9/CFwzv4wkEGOb2Fr8Bb+MYgg4SEDBJSskhJyCAnd0oGCQkZJNzgA4vAPXzkcA+fWH6Hz8Bv1B888j38ybP38IXDO/jCQQY5vYWvwFv4xiCDhIQMElKySEnIICd3SgYJCRkk3OADi3APH3l0D59YfofPwG/wmUcy+B7+5Nl7+MKjd/AFkEF+eAtfeQt8Y5BBQkIGCSlZpCRkkJM7JYOEhAwSbsAHXgj38JFH9/CJF78Dn4Hf4DOPZPA9/Mmz9/CFR+/gCyAvfr3JD2/hK2/hG5MMEhIySEjJIiUhg5zcKRkkJGSQkA0+8EK4h488uodPvPgdPnP4DT7zSAbfw588ew9fePQOvgDy7Ff4yg9vga9v4RuTDBISMkhIySIlIYOc3CkZJCRkkJANPvBC7uEjT+7hEy9+h88cfgM+c5BH//qDR76HP3n2Hr7w5B18AXn2K3zl2VuefGOSQUJCBgkpWaQkZJCTOyWDhIQMEsIGH3jmPfCRJ/fwiWe/A5959Bt85iCHf8EfHHwPf/LsPXzhyTvgC/LDr3CTZ2959I2QQUJCBgkpWaQkZJCTOyWDhIQMEsIGfOCHDfjI6R74xOl34DNPfgM+AwL/Av7g4Hv4k2fv4Qund8AXOf0K3OTZWx59I2SQkJBBQkoWKQkZ5OROySAhIYOEwMbhA4824CM/3PPoE4ffgU9y+o3DZ4R/AX/wyPfwJ8/ewxd+eAfcePIrcENevAW+SsggISGDhJQsUhIyyMmdkkFCQgYJOWxMH/nhnvgkP/zG8gdPfA9/8uw9fOGHd0w3kBdvga8SMkhIyCAhJYuUhAxycqdkkJCQQUIebSwfeXbP9Al59hsv/uCJ7+FPnr2HLzx7x3IDZHkLXyVkkJCQQUJKFikJGeTkTskgISGDhJw2nnyQ5R4+8TuPPnGQ5Tce/cEzef8ny/svTO94cuORDG+/IiGDhIQMElKySEnIICd3SgYJCRkkpGS5h08MEjJISMggJSGDhIQMElKySEnIICd3SgYJCRkkpGS5h08MEjJISMggJSGDhIQMElKySEnIICd3SgYJCRkkpGS5h08MEjJISMggJSGDhIQMElKySEnIICd3SgYJCRkkpGS5h08MEjJISMggJSGDhIQMElKySEnIICd3SgYJCRkkpGS5h08MEjJISMggJSGDhIQMElKySEnIICd3SgYJCRkkpGS5h08MEjJISMggJSGDhIQMElKySEnIICd3SgYJCRkkpGS5h08MEjJISMggJSGDhIQMElKySEnIICd3SgYJCRkkpGTx909MEjJISMggJSGDhIQMElKySEnIICd3SgYJCRkkpGSRkpBBQkIGKQkZJCRkkJCSRUpCBjm5UzJISMggISWLlIQMEhIySEnIICEhg4SULFISMsjJnZJBQkIGCSlZpCRkkJCQQUpCBgkJGSSkZJGSkEFO7pQMEhIySEjJIiUhg4SEDFISMkhIyCAhJYuUhAxycqdkkJCQQUJKFikJGSQkZJCSkEFCQgYJKVmkJGSQkzslg4SEDP+nKjgmABiIgRhmky1/CB1+yFmSkJIjJSFDQkKGlIQMCQkZElJypCRkyONHyZCQkCEhJUdKQoaEhAwpCRkSEjIkpORISciQx4+SISEhQ0JKjpSEDAkJGVISMiQkZEhIyZGSkCGPHyVDQkKGhJQcKQkZEhIypCRkSEjIkJCSIyUhQx4/SoaEhAwJKTlSEjIkJGRISciQkJAhISVHSkKGPH6UDAkJGRJScqQkZEhIyJCSkCEhIUNCSo6UhAx5/CgZEhIyJKTkSEnIkJCQISUhQ0JChoSUHCkJGfL4UTIkJGRISMmRkpAhISFDSkKGhIQMCSk5UhIy5PGjZEhIyJCQkiMlIUNCQoaUhAwJCRkSUnKkJGTI40fJkJCQISElR0pChoSEDCkJGRISMiSk5EhJyJDHj5IhISFDQkqOlIQMCQkZUhIyJCRkSEjJkZKQIY8fJUNCQoaElBwpCRkSEjKkJGRISMiQkJIjJSFDHj9KhoSEDAkpOVISMiQkZEhJyJCQkCEhJUdKQoY8fpQMCQkZElJypCRkSEjIkJKQISEhQ0JKjpSEDHn8KBkSEjIkpORISciQkJAhJSFDQkKGhJQcKQkZ8vhRMiQkZEhIyZGSkCEhIUNKQoaEhAwJKTlSEjLk+QGer4wf5k9wzAAAAABJRU5ErkJggg==', 'gurultulu.png': '...', 'temiz.png': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAAAAACfHP67AAAgAElEQVR4AbzBAQEAIADDINbmZc1gVVsIjUgkEolIJBKJSCQSiUgkEolIJBKJSCQSiUQkEolEJBKJRCQSiUQkEolEJBKJRCISiUQiEolEIhKJRCISiUQiEolEIjUikUgkIpFIJCKRSCQikUgkIpFIJCKRSCQSkUgkEpFIJBKRSCQSkUgkEpFIJBKJSCQSiUgkEolIJBKJSCQSiUgkEonUiEQikYhEIpGIRCKRiEQikYhEIpGIRCKRSEQikUhEIpFIRCKRSEQikUhEIpFIJCKRSCQikUgkIpFIJCKRSCQikUgkUiMSiUQiEolEIhKJRCISiUQiEolEIhKJRCIRiUQiEYlEIhGJRCIRiUQiEYlEIpGIRCKRiEQikYhEIpGIRCKRiEQikUiNSCQSiUgkEolIJBKJSCQSiUgkEolIJBKJRCQSiUQkEolEJBKJRCQSiUQkEolEIhKJRCISiUQiEolEIhKJRCISiUQiNSKRSCQikUgkIpFIJCKRSCQikUgkIpFIJBKRSCQSkUgkEpFIJBKRSCQSkUgkEolIJBKJSCQSiUgkEolIJBKJSCQSidSIRCKRiEQikYhEIpGIRCKRiEQikYhEIpFIRCKRSEQikUhEIpFIRCKRSEQikUgkIpFIJCKRSCQikUgkIpFIJCKRSCRSIxKJRCISiUQiEolEIhKJRCISiUQiEolEIhGJRCIRiUQiEYlEIhGJRCIRiUQikYhEIpGIRCKRiEQikYhEIpGIRCKRSI1IJBKJSCQSiUgkEolIJBKJSCQSiUgkEolEJBKJRCQSiUQkEolEJBKJRCQSiUQiEolEIhKJRCISiUQiEolEIhKJRCI1IpFIJCKRSCQikUgkIpFIJCKRSCQikUgkEpFIJBKRSCQSkUgkEpFIJBKRSCQSiUgkEolIJBKJSCQSiUgkEolIJBKJ1IhEIpGIRCKRiEQikYhEIpGIRCKRiEQikUhEIpFIRCKRSEQikUhEIpFIRCKRSCQikUgkIpFIJCKRSCQikUgkIpFIJFIjEolEIhKJRCISiUQiEolEIhKJRCISiUQiEYlEIhGJRCIRiUQiEYlEIhGJRCKRiEQikYhEIpGIRCKRiEQikYhEIpFIjUgkEolIJBKJSCQSiUgkEolIJBKJSCQSiUQkEolEJBKJRCQSiUQkEolEJBKJRCISiUQiEolEIhKJRCISiUQiEolEIjUikUgkIpFIJCKRSCQikUgkIpFIJCKRSCQSkUgkEpFIJBKRSCQSkUgkEpFIJBKJSCQSiUgkEolIJBKJSCQSiUgkEonUiEQikYhEIpGIRCKRiEQikYhEIpGIRCKRSEQikUhEIpFIRCKRSEQikUhEIpFIJCKRSCQikUgkIpFIJCKRSCQikUgkUiMSiUQiEolEIhKJRCISiUQiEolEIhKJRCIRiUQiEYlEIhGJRCIRiUQiEYlEIpGIRCKRiEQikYhEIpGIRCKRiEQikUiNSCQSiUgkEolIJBKJSCQSiUgkEolIJBKJRCQSiUQkEolEJBKJRCQSiUQkEolEIhKJRCISiUQiEolEIhKJRCISiUQiNSKRSCQikUgkIpFIJCKRSCQikUgkIpFIJBKRSCQSkUgkEpFIJBKRSCQSkUgkEolIJBKJSCQSiUgkEolIJBKJSCQSidSIRCKRiEQikYhEIpGIRCKRiEQikYhEIpFIRCKRSEQikUhEIpFIRCKRSEQikUgkIpFIJCKRSCQikUgkIpFIJCKRSCRSIxKJRCISiUQiEolEIhKJRCISiUQiEolEIhGJRCIRiUQiEYlEIhGJRCIRiUQikYhEIpGIRCKRiEQikYhEIpGIRCKRSI1IJBKJSCQSiUgkEolIJBKJSCQSiUgkEolEJBKJRCQSiUQkEolEJBKJRCQSiUQiEolEIhKJRCISiUQiEolEIhKJRCI1IpFIJCKRSCQikUgkIpFIJCKRSCQikUgkEpFIJBKRSCQSkUgkEpFIJBKRSCQSiUgkEolIJBKJSCQSiUgkEolIJBKJ1IhEIpGIRCKRiEQikYhEIpGIRCKRiEQikUhEIpFIRCKRSEQikUhEIpFIRCKRSCQikUgkIpFIJCKRSCQikUgkIpFIJFIjEolEIhKJRCISiUQiEolEIhKJRCISiUQiEYlEIhGJRCIRiUQiEYlEIhGJRCKRiEQikYhEIpGIRCKRiEQikYhEIpFIjUgkEolIJBKJSCQSiUgkEolIJBKJSCQSiUQkEolEJBKJRCQSiUQkEolEJBKJRCISiUQiEolEIhKJRCISiUQiEolEIjUikUgkIpFIJCKRSCQikUgkIpFIJCKRSCQSkUgkEpFIJBKRSCQSkUgkEpFIJBKJSCQSiUgkEolIJBKJSCQSiUgkEonUiEQikYhEIpGIRCKRiEQikYhEIpGIRCKRSEQikUhEIpFIRCKRSEQikUhEIpFIJCKRSCQikUgkIpFIJCKRSCQikUgkUiMSiUQiEolEIhKJRCISiUQiEolEIhKJRCIRiUQiEYlEIhGJRCIRiUQiEYlEIpGIRCKRiEQikYhEIpGIRCKRiEQikUiNSCQSiUgkEolIJBKJSCQSiUgkEolIJBKJRCQSiUQkEolEJBKJRCQSiUQkEolEIhKJRCISiUQiEolEIhKJRCISiUQiNSKRSCQikUgkIpFIJCKRSCQikUgkIpFIJBKRSCQSkUgkEpFIJBKRSCQSkUgkEolIJBKJSCQSiUgkEolIJBKJSCQSidSIRCKRiK6vjohEIpGIRCKRiEQikYhEIpGIRCKRSEQikUhEIpFIRCKRSEQikUhEIpFIpEYkEolEdH11RCQSiUQkEolEJBKJRCQSiUQkEolEIhKJRCISiUQiEolEIhKJRCISiUQiNSKRSCSi66sjIpFIJCKRSCQikUgkIpFIJCKRSCQSkUgkEpFIJBKRSCQSkUgkEpFIJBKpEYlEIhFdXx0RiUQiEYlEIhGJRCIRiUQiEYlEIpGIRCKRiEQikYhEIpGIRCKRiEQikUiNSCQSiej66ohIJBKJSCQSiUgkEolIJBKJSCQSiUQkEolEJBKJRCQSiUQkEolEJBKJRGpEIpFIRNdXR0QikUhEIpFIRCKRSEQikUhEIpFIJCKRSCQikUgkIpFIJCKRSCQikUgkUiMSiUQiur46IhKJRCISiUQiEolEIhKJRCISiUQiEYlEIhGJRCIRiUQiEYlEIhGJRCKRGpFIJBLR9dURkUgkEpFIJBKRSCQSkUgkEpFIJBKJSCQSiUgkEolIJBKJSCQSiUgkEonUiEQikYiur46IRCKRiEQikYhEIpGIRCKRiEQikUhEIpFIRCKRSEQikUhEIpFIRCKRSKRGJBKJRHR9dUQkEolEJBKJRCQSiUQkEolEJBKJRCISiUQiEolEIhKJRCISiUQiEolEIjUikUgkouurIyKRSCQikUgkIpFIJCKRSCQikUgkEpFIJBKRSCQSkUgkEpFIJBKRSCQSqRGJRCIRXV8dEYlEIhGJRCIRiUQiEYlEIhGJRCKRiEQikYhEIpGIRCKRiEQikYhEIpFIjUgkEono+uqISCQSiUgkEolIJBKJSCQSiUgkEolEJBKJRCQSiUQkEolEJBKJRCQSiURqRCKRSETXV0dEIpFIRCKRSEQikUhEIpFIRCKRSCQikUgkIpFIJCKRSCQikUgkIpFIJFIjEolEIrq+OiISiUQiEolEIhKJRCISiUQiEolEIhGJRCIRiUQiEYlEIhGJRCIRiUQikRqRSCQS0fXVEZFIJBKRSCQSkUgkEpFIJBKRSCQSiUgkEolIJBKJSCQSiUgkEolIJBKJ1IhEIpGIrq+OiEQikYhEIpGIRCKRiEQikYhEIpFIRCKRSEQikUhEIpFIRCKRSEQikUikRiQSiUR0fXVEJBKJRCQSiUQkEolEJBLpEQd3x2EdBQBGz9fNdsIDA0wgIY5jS7ehrWH74IGBgZCQH9vSbYgGmIEk0u45JCKRSCQSkUgkEpFIJBKRSCQSkUgkEpFIJBKpQSQSiUS0bDVFJBKJRCQSiUQkEolEJBKJRCQSiUQiEolEIhKJRCISiUQiEolEIhKJRCI1iEQikYiWraaIRCKRiEQikYhEIpGIRCKRiEQikUhEIpFIRCKRSEQikUhEIpFIRCKRSKQGkUgkEtGy1RSRSCQSkUgkEpFIJBKRSCQSkUgkEolIJBKJSCQSiUgkEolIJBKJSCQSidQgEolEIlq2miISiUQiEolEIhKJRCISiUQiEolEIhGJRCIRiUQiEYlEIhGJRCIRiUQikRpEIpFIRMtWU0QikUhEIpFIRCKRSEQikUhEIpFIJCKRSCQikUgkIpFIJCKRSCQikUgkUoNIJBKJaNlqikgkEolIJBKJSCQSiUgkEolIJBKJRCQSiUQkEolEJBKJRCQSiUQkEolEahCJRCIRLVtNEYlEIhGJRCIRiUQiEYlEIhGJRCKRiEQikYhEIpGIRCKRiEQikYhEIpFIDSKRSCSiZaspIpFIJCKRSCQikUgkIpFIJCL5y29EIpGIRCKRiEQikYhEIpGIRCKRiEQikUgNIpFIJKJlqykikUgkIpFIJCKRSCQikUj0V//VbyMRiUQiEYlEIhGJRCIRiUQiEYlEIpEaRCKRSETLVlNEIpFIRCKRSEQikUhEItHf/A+/i0gkEolIJBKJSCQSiUgkEolIJBKJ1CASiUQiWraaIhKJRCISiUQiEolEIhLp7/4vv49EIpGIRCKRiEQikYhEIpGIRCKRSA0ikUgkomWrKSKRSCQikUgkIpFIJCLxDz/DHxKJRCISiUQiEolEIhKJRCISiUQiNYhEIpGIlq2miEQikYhEIpGIRCKRiPzTz/YZiUQiEolEIhKJRCISiUQiEolEIjWIRCKRiJatpohEIpGIRCKRiEQikYhv/CKfJRKJSCQSiUgkEolIJBKJSCQSidQgEolEIlq2miISiUQiEolEIhKJRKJv/GJ/jEQiEolEIhKJRCISiUQiEolEIjWIRCKRiJatpohEIpGIRCKRiEQikfzLr/IniUQkEolEJBKJRCQSiUQkEolEahCJRCIRLVtNEYlEIhGJRCIRiUQi3/rVPheJSCQSiUgkEolIJBKJSCQSidQgEolEIlq2miISiUQiEolEIhKJRN96AZ9HIhKJRCISiUQiEolEIhKJRCI1iEQikYiWraaIRCKRiEQikYhEIvnOC/lCIhKJRCISiUQiEolEIhKJRCI1iEQikYiWraaIRCKRiEQikYhEIn3nxXyRiEQikYhEIpGIRCKRiEQikUgNIpFIJKJlqykikUgkIpFIJCKR6N9e1J8jEolEIhKJRCISiUQiEolEIjWIRCKRiJatpohEIpGIRCKRiETiey/sSyKRSCQikUgkIpFIJCKRSCRSg0gkEolo2WqKSCQSiUgkEolIJN97cV+KRCKRiEQikYhEIpGIRCKRSA0ikUgkomWrKSKRSCQikUgkIpG+9wq+jEQikYhEIpGIRCKRiEQikUgNIpFIJKJlqykikUgkIpFIJCKRH7yKNyQSiUQkEolEJBKJRCQSiURqEIlEIhEtW00RiUQiEYlEIhGJfvBK3kgkEolIJBKJSCQSiUgkEonUIBKJRCJatpoiEolEIhKJRCISP3o1bxKJRCISiUQiEolEIhKJRCI1iEQikYiWraaIRCKRiEQikYjkR6/oK5FIJCKRSCQikUgkIpFIJFKDSCQSiWjZaopIJBKJSCQSiUg/elVfRSKRiEQikYhEIpGIRCKRSA0ikUgkomWrKSKRSCQikUgkIj95ZW9JJBKRSCQSkUgkEpFIJBKpQSQSiUS0bDVFJBKJRCQSiUT0k1f3ViKRiEQikYhEIpGIRCKRSA0ikUgkomWrKSKRSCQikUgk4oMN3iYSiUgkEolIJBKJSCQSidQgEolEIlq2miISiUQiEolEIh9s8bVIJCKRSCQikUgkIpFIJFKDSCQSiWjZaopIJBKJSCQSiT7Y5OtIJCKRSCQikUgkIpFIJFKDSCQSiWjZaopIJBKJSCQSiT7Y5OtIJCKRSCQikUgkIpFIJFKDSCQSiWjZaopIJBKJSCQSiY+2eUciEYlEIhGJRCIRiUQikRpEIpFIRMtWU0QikUhEIpFIPtronUQiEolEIhKJRCISiUQiNYhEIpGIlq2miEQikYhEIpF8tNE7iUQkEolEJBKJRCQSiURqEIlEIhEtW00RiUQiEYlEIn201btEIhKJRCISiUQiEolEIjWIRCKRiJatpohEIpGIRCKRTzZ7LxKRSCQSkUgkEpFIJBKpQSQSiUS0bDVFJBKJRCQSiXyy2XuRiEQikYhEIpGIRCKRSA0ikUgkomWrKSKRSCQikUj0yXbvIxGJRCIRiUQiEYlEIpEaRCKRSETLVlNEIpFIRCKR6JPt3kciEolEIhKJRCISiUQiNYhEIpGIlq2miEQikYhEIvHkgAcSkUgkEpFIJBKRSCQSqUEkEolEtGw1RSQSiUQkEoknBzyQiEQikYhEIpGIRCKRSA0ikUgkomWrKSKRSCQikUieHPEgEYlEIhGJRCIRiUQikRpEIpFIRMtWU0QikUhEIpE8OeJBIhKJRCISiUQiEolEIjWIRCKRiJatpohEIpGIRCI9OeQhEYlEIhGJRCIRiUQikRpEIpFIRMtWU0QikUhEIpGeHPKQiEQikYhEIpGIRCKRSA0ikUgkomWrKSKRSCQikUhPDnlIRCKRSEQikUhEIpFIpAaRSCQS0bLVFJFIJBKRSOTZMY8iEolEIhKJRCISiUQiNYhEIpGIlq2miEQikYhEIs+OeRSRSCQSkUgkEpFIJBKpQSQSiUS0bDVFJBKJRCQSeXbMo4hEIpGIRCKRiEQikUgNIpFIJKJlqykikUgkIpHo2UGPEYlEIhGJRCIRiUQikRpEIpFIRMtWU0QikUhEItGzgx4jEolEIhKJRCISiUQiNYhEIpGIlq2miEQikYhEomcHPUYkEolEJBKJRCQSiURqEIlEIhEtW00RiUQiEYnE7aiLSCQSiUgkEolIJBKJ1CASiUQiWraaIhKJRCISidtRF5FIJBKRSCQSkUgkEqlBJBKJRLRsNUUkEolEJBK3oy4ikUgkIpFIJCKRSCRSg0gkEolo2WqKSCQSiUgkbkddRCKRSEQikUhEIpFIpAaRSCQS0bLVFJFIJBKRSNyOuohEIpGIRCKRiEQikUgNIpFIJKJlqykikUgkIpHcDrtEIpFIRCKRSEQikUikBpFIJBLRstUUkUgkEpFIboddIpFIJCKRSCQikUgkUoNIJBKJaNlqikgkEolIJLfDLpFIJBKRSCQSkUgkEqlBJBKJRLRsNUUkEolEJJLbYZdIJBKJSCQSiUgkEonUIBKJRCJatpoiEolEIhLJ7bBLJBKJRCQSiUQkEolEahCJRCIRLVtNEYlEIhGJ5HbYJRKJRCISiUQiEolEIjWIRCKRiJatpohEIpGIRHI77BKJRCIRiUQiEYlEIpEaRCKRSETLVlNEIpFIRCK5HXaJRCKRiEQikYhEIpFIDSKRSCSiZaspIpFIJCKR3A67RCKRSEQikUhEIpFIpAaRSCQS0bLVFJFIJBKRSG6HXSKRSCQikUgkIpFIJFKDSCQSiWjZaopIJBKJSCS3wy6RSCQSkUgkEpFIJBKpQSQSiUS0bDVFJBKJRCTS7bgrEolEIhKJRCISiUQiNYhEIpGIlq2miEQikYhEcjvsEolEIhGJRCIRiUQikRpEIpFIRMtWU0QikUhEIrkddolEIpGIRCKRiEQikUgNIpFIJKJlqykikUgkIpHcDrtEIpFIRCKRSEQikUikBpFIJBLRstUUkUgkEpFIboddIpFIJCKRSCQikUgkUoNIJBKJaNlqikgkEolIJLfDLpFIJBKRSCQSkUgkEqlBJBKJRLRsNUUkEolEJJLbYZdIJBKJSCQSiUgkEonUIBKJRCJatpoiEolEIhLJ7bBLJBKJRCQSiUQkEolEahCJRCIRLVtNEYlEIhGJ5HbYJRKJRCISiUQiEolEIjWIRCKRiJatpohEIpGIRHI77BKJRCIRiUQiEYlEIpEaRCKRSETLVlNEIpFIRCK5HXaJRCKRiEQikYhEIpFIDSKRSCSiZaspIpFIJCKR3A67RCKRSEQikUhEIpFIpAaRSCQS0bLVFJFIJBKRSNyOuohEIpGIRCKRiEQikUgNIpFIJKJlqykikUgkIpG4HXURiUQiEYlEIhGJRCKRGkQikUhEy1ZTRCKRSEQicTvqIhKJRCISiUQiEolEIjWIRCKRiJatpohEIpGIROJ21EUkEolEJBKJRCQSiURqEIlEIhEtW00RiUQiEYnE7aiLSCQSiUgkEolIJBKJ1CASiUQiWraaIhKJRCISiZ4d9BiR/kMcvONoQQUAGD3fbu5OLIwaFOQ5M5V2dFS6A5JLR333YWE0iiCPea3IgoLGRIH5738OiUQiEolEIhKJRCI1iEQikYiWraaIRCKRiESia0d0FpFIJBKRSCQSkUgkEqlBJBKJRLRsNUUkEolEJBJdO6KziEQikYhEIpGIRCKRSA0ikUgkomWrKSKRSCQikci1ozkTkUgkEpFIJBKRSCQSqUEkEolEtGw1RSQSiUQkErl2NGciEolEIhKJRCISiUQiNYhEIpGIlq2miEQikYhEIteO5kxEIpFIRCKRSEQikUikBpFIJBLRstUUkUgkEpFIpCtHcpqIRCKRiEQikYhEIpFIDSKRSCSiZaspIpFIJCKRSFeO5DQRiUQiEYlEIhGJRCKRGkQikUhEy1ZTRCKRSEQika4cyWkiEolEIhKJRCISiUQiNYhEIpGIlq2miEQikYhEIrlyFKcSkUgkEpFIJBKRSCQSqUEkEolEtGw1RSQSiUQkEsmVoziViEQikYhEIpGIRCKRSA0ikUgkomWrKSKRSCQikUhcOYJTEpFIJBKRSCQSkUgkEqlBJBKJRLRsNUUkEolEJBKJK0dwSiISiUQiEolEIhKJRCI1iEQikYiWraaIRCKRiEQi0aXtTiIRiUQiEYlEIhGJRCKRGkQikUhEy1ZTRCKRSEQikejSdieRiEQikYhEIpGIRCKRSA0ikUgkomWrKSKRSCQikUjk0mYnIhGJRCIRiUQiEYlEIpEaRCKRSETLVlNEIpFIRCKRyKXNTkQiEolEIhKJRCISiUQiNYhEIpGIlq2miEQikYhEIpEubPUokYhEIpGIRCKRiEQikUgNIpFIJKJlqykikUgkIpFIJBc2eiSRiEQikYhEIpGIRCKRSA0ikUgkomWrKSKRSCQikUgkFzZ6JJGIRCKRiEQikYhEIpFIDSKRSCSiZaspIpFIJCKRSCQubPOIRCISiUQiEolEIhKJRCI1iEQikYiWraaIRCKRiEQikejcJg8jkYhEIpGIRCKRiEQikUgNIpFIJKJlqykikUgkIpFIJDq3ycNIJCKRSCQikUgkIpFIJFKDSCQSiWjZaopIJBKJSCQSiZzb4qFIJCKRSCQikUgkIpFIJFKDSCQSiWjZaopIJBKJSCQSiTi3wYNEIhGJRCIRiUQiEYlEIpEaRCKRSETLVlNEIpFIRCKRSETvHNwDiUQiEolEIhKJRCISiUQiNYhEIpGIlq2miEQikYhEIpGIvHNgD0gkEpFIJBKRSCQSkUgkEqlBJBKJRLRsNUUkEolEJBKJRKS3Dup+JBKJSCQSiUgkEolIJBKJ1CASiUQiWraaIhKJRCISiUQikrcO6L5IJBKRSCQSkUgkEpFIJBKpQSQSiUS0bDVFJBKJRCQSiUQk3jqYe4lEIhGJRCIRiUQiEYlEIpEaRCKRSETLVlNEIpFIRCKRSESiNw7knkQikYhEIpGIRCKRiEQikUgNIpFIJKJlqykikUgkIpFIJCKRNw7iHolEIhGJRCIRiUQiEYlEIpEaRCKRSETLVlNEIpFIRCKRSEQivXYAdyORSCQikUgkIpFIJCKRSCRSg0gkEolo2WqKSCQSiUgkEolIJK/duLsikUgkIpFIJCKRSCQikUgkUoNIJBKJSCQSiUgkEolIJBKJSCQSiUgkEolEJBKJRCQSiUQkEq/dsLtEIpFIRCKRSEQikUhEIpFIpAaRSCQSkUgkEpFIJBKRSCQSkUgkEpFIJBKJSCQSiUgkEolIJPrbjfo+IpFIJCKRSCQikUgkIpFIJFKDSCQSiUgkEolIJBKJSCQSiUgkEolIJBKJRCQSiUQkEolEJBLplRtzJxGJRCIRiUQiEYlEIhGJRCKRGkQikUhEIpFIRCKRSEQikUhEIpFIRCKRSCQikUgkIpFIJCKRSF65IXckIpFIJCKRSCQikUgkIpFIJFKDSCQSiUgkEolIJBKJSCQSiUgkEolIJBKJRCQSiUQkEolEJBKJXroBtyMRiUQiEYlEIhGJRCIRiUQikRpEIpFIRCKRSEQikUhEIpFIRCKRSEQikUgkIpFIJCKRSCQikUjkpc92WyQikUgkIpFIJCKRSCQikUgkUoNIJBKJSCQSiUgkEolIJBKJSCQSiUgkEolEJBKJRCQSiUQkEonkL5/lO4lEJBKJRCQSiUQkEolEJBKJRGoQiUQiEYlEIhGJRCIRiUQiEYlEIhGJRCKRiEQikYhEIpGIRCKR6IVP9m0kEpFIJBKRSCQSkUgkEpFIJBKpQSQSiUQkEolEJBKJRCQSiUQkEolEJBKJRCISiUQiEolEIhKJRCJe+CS3EolEJBKJRCQSiUQkEolEJBKJRGoQiUQiEYlEIhGJRCIRiUQiEYlEIhGJRCKRiEQikYhEIpGIRCKRiPzpo90ikUhEIpFIRCKRSEQikUhEIpFIpAaRSCQSkUgkEpFIJBKRSCQSkUgkEpFIJBKJSCQSiUgkEolIJBKJSPzhI3yTSCQSkUgkEpFIJBKRSCQSkUgkEqlBJBKJRCQSiUQkEolEJBKJRCQSiUQkEolEIhKJRCISiUQiEolEIhLpd//L1+Q5AMMAABUKSURBVJFIJBKRSCQSkUgkEpFIJBKRSCQSqUEkEolEJBKJRCQSiUQkEolEJBKJRCQSiUQiEolEIhKJRCISiUQiEol+8x++ikgkEolIJBKJSCQSiUgkEolIJBKJ1CASiUQiEolEIhKJRCISiUQiEolEIhKJRCIRiUQiEYlEIhGJRCIRiUSiX/2rLyMRiUQiEYlEIhGJRCIRiUQiEYlEIpEaRCKRSEQikUhEIpFIRCKRSEQikUhEIpFIJCKRSCQikUgkIpFIJCKRSCQi+eULkUgkIpFIJCKRSCQikUgkIpFIJCKRSCRSg0gkEolIJBKJSCQSiUgkEolIJBKJSCQSiUQkEolEJBKJRCQSiUQkEolEJBKJRCISiUQiEolEIhKJRCISiUQiEolEIjWIRCKRiEQikYhEIpGIRCKRiEQikYhEIpFIRCKRSEQikUhEIpFIRCKRSEQikUgkIpFIJCKRSCQikUgkIpFIJCKRSCRSg0gkEolIJBKJSCQSiUgkEolIJBKJSCQSiUQkEolEJBKJRCQSiUQkEolEJBKJRCISiUQiEolEIhKJRCISiUQiEolEIjWIRCKRiEQikYhEIpGIRCKRiEQikYhEIpFIRCKRSEQikUhEIpFIRCKRSEQikUgkIpFIJCKRSCQikUgkIpFIJCKRSCRSg0gkEolIJBKJSCQSiUgkEolIJBKJSCQSiUQkEolEJBKJRCQSiUQkEolEJBKJRCISiUQiEolEIhKJRCISiUQiEolEIjWIRCKRiEQikYhEIpGIRCKRiEQikYhEIpFIRCKRSEQikUhEIpFIRCKRSEQikUgkIpFIJCKRSCQikUgkIpFIJCKRSCRSg0gkEolIJBKJSCQSiUgkEolIJBKJSCQSiUQkEolEJBKJRCQSiUQkEolEJBKJRCISiUQiEolEIhKJRCISiUQiEolEIjWIRCKRiEQikYhEIpGIRCKRiEQikYhEIpFIRCKRSEQikUhEIpFIRCKRSEQikUgkIpFIJCKRSCQikUgkIpFIJCKRSCRSg0gkEolIJBKJSCQSiUgkEolIJBKJSCQSiUQkEolEJBKJRCQSiUQkEolEJBKJRCISiUQiEolEIhKJRCISiUQiEolEIjWIRCKRiEQikYhEIpGIRCKRiEQikYhEIpFIRCKRSEQikUhEIpFIRCKRSEQikUgkIpFIJCKRSCQikUgkIpFIJCKRSCRSg0gkEolIJBKJSCQSiUgkEolIJBKJSCQSiUQkEolEJBKJRCQSiUQkEolEJBKJRCISiUQiEolEIhKJRCISiUQiEolEIjWIRCKRiEQikYhEIpGIRCKRiEQikYhEIpFIRCKRSEQikUhEIpFIRCKRSEQikUgkIpFIJCKRSCQikUgkIpFIJCKRSCRSg0gkEolIJBKJSCQSiUgkEolIJBKJSCQSiUQkEolEJBKJRCQSiUQkEolEJBKJRCISiUQiEolEIhKJRCISiUQiEolEIjWIRCKRiEQikYhEIpGIRCKRWN77QSQSkUgkEolIJBKJSCQS+cl7T0k88YxEJB57TiIRiUQikYhEIpGIRCKRiEQikYhEIpGIRCKRSA0ikUgkIpFIJCKRSCQikUik5YMfSSQikUgkEpFIJBKRSCQ/++Cp5AnPSETiMc8lEpFIJBKJSCQSyT9cwY1uU4cBgNHzvc19rKSakqgucf4awrAnW9ulg4CJE9IrGaThN12C1kF7jkgkEolIJBKJSCQSiUgkEonUQCQSiUQkEolEJBKJRCQSiXz1oyOJRCQSiUQiEolEIhKJ9NqP3iVX2EhE4gWPEolIJBKJRCQSiUQkEolEJBKJRCQSiUQkEolEaiASiUQiEolEIhKJRCISiURf8RPJF0+OEolIJBKJRCQSiUQkEll48hvRS9yKKzYSkbzgkUQiEolEIhGJRCIRiUQiEYlEIhGJRCIRiUQikRqIRCKRiEQikYhEIpGIRCKxx2EkvuAokYhEIpFIRCKRSEQikQXeROQlbuOKjUQkL3gkkYhEIpFIRCKRSEQikUhEIpFIRCKRSEQikUikBiKRSCQikUgkIpFIJCKRSOw5FIk+41gkIpFIJBKRSCQSkUi0wBsR6Zrb6JJNItILPJJIRCKRSCQikUgkIpFIJCKRSCQikUgkIpFIJFIDkUgkEpFIJBKRSCQSkUgkew6JRD7jOBKRSCQSiUgkEolIJJYYE5Fc854u2SQiveBTJBKRSCQSiUgkEolIJBKJSCQSiUgkEolIJBKJ1EAkEolEJBKJRCQSiUQkEsmeQyKRdhxHIhKJRCIRiUQiEYnEklEikmve0yU+ikgv+BSJRCQSiUQiEolEIhKJRCISiUQiEolEIhKJRCI1EIlEIhGJRCIRiUQiEYlE2nMQkUg7jiP9/nM8enIqEom89+ylSGRcyD89W4nEklEiEte3kUs+isgpn0QiEYlEIpGIRCKRiEQikYhEIpGIRCKRiEQikUgNRCKRSEQikUhEIpFIRCKR9hxEJNIOJ2Ri9uh/5iQSH/zhhkQji3/5w5pkiVEiEolc8lFETvkkEolIJBKJRCQSiUQkEolEJBKJRCQSiUQkEolEaiASiUQiEolEIhKJRCISibTnICKR7Dghkx/NJZIPvruRaPQna8mSkUQkErnEx4hO8UkkEpFIJBKJSCQSiUgkEolIJBKJSCQSiUgkEonUQCQSiUQkEolEJBKJRCQSac9BRCLZcUImz36RB0/mifTBk1/lnSc3iUbf/EPWWEuWjCQikUgXfIzolIdEIhGJRCKRiEQikYhEIpGIRCKRiEQikYhEIpFIDUQikUhEIpFIRCKRSEQikfYcRCSSHSdk8mSW6B5nIhtcRfSWm0SjJ8uINWvJkpFEJBLpAndEpzwkEolIJBKJRCQSiUQkEolEJBKJRCQSiUQkEolEaiASiUQiEolEIhKJRCISibTnICKR7DghE2YSucdZZIOriLzllWjEUsQa62TJSCISieSCO2KOh0QiEYlEIpGIRCKRiEQikYhEIpGIRCKRiEQikUgNRCKRSEQikUhEIpFIRCKR9hxEJJIdJ2RiRiJtOYs2uBKRt7wSjSwS0Yp1smQkEYlEcsEdMedBIpGIRCKRSEQikUhEIpFIRCKRSEQikUhEIpFIpAYikUgkIpFIJCKRSCQikUh7DiISyY4TMjEjkbacRRt/8Uo0skhEK9bJkpFEJBLJBe7EnAeJRCISiUQiEYlEIhGJRCIRiUQiEYlEIhGJRCKRGohEIpGIRCKRiEQikYhEIu05iEgkO07IxIxE2uKcNv7ilWhkkYhWrJMlI4lIJBIX3MmcBxKJRCQSiUQiEolEIhKJRCISiUQiEolEIhKJRCI1EIlEIhGJRCIRiUQiEYlE2nMQkUh2nJCJGYlkyzlt/NlNopFFIlqxTpaMJCKRSFxwJ3MeSCQSkUgkEolIJBKJSCQSiUgkEolIJBKJSCQSidRAJBKJRCQSiUQkEolEJBJpz0FEIu04jkzMSCRbzmnDZSISiUYWiWjFOrFklIhEItE5d83xQCKRiEQikUhEIpFIRCKRSEQikUhEIpFIRCKRSKQGIpFIJCKRSCQikUgkIpFIexwSibTjODIxI5G2OKcNl4lIJBpZJKIV68SSUSIS17eR6Jy75txHIpGIRCKRSEQikUhEIpFIRCKRSEQikUhEIpFIpAYikUgkIpFIJCKRSCQikUj2HBKJtOM4MjEjkbacRRsuE5FINLJIRCvWiSWjRCTXvCfRObZz7iORSEQikUgkIpFIJCKRSCQikUgkIpFIJCKRSCRSA5FIJBKRSCQSkUgkEpFIJHsOiUQ+cywyMSORtpxFG1yJSP++STSySEQr1okWjIlIrnlPIue+uY9EIhGJRCKRiEQikYhEIpGIRCKRiEQikYhEIpFIDUQikUhEIpFIRCKRSEQikdhzKBJ9xrHIxIxE7nEW2XAlIm95JRpZJKIV60QLvBGRrrmNRM49uxeJRCISiUQiEYlEIhGJRCIRiUQiEYlEIhGJRCKRGohEIpGIRCKRiEQikYhEIrHHYSS+4CiRCTOJ7nEmssFVRG/xSjSySEQr1oks8CYiL3EbiZx7di8SiUQkEolEIhKJRCISiUQiEolEIhKJRCISiUQiNRCJRCIRiUQiEYlEIhGJRKKvnvwk+YIjiUye/SIeME+kD578Kt7hJtHIIhGtWCfSa/xG9BK3IpHOsE0kEolIJBKJRCQSiUQkEolEJBKJRCQSiUQkEolEaiASiUQiEolEIhKJRCISiUS++tGRRCZ/Mk8kH3x3I9HIIhGtWCfSaz96l0ikM2wTiUQiEolEIhGJRCIRiUQiEYlEIhGJRCIRiUQikRqIRCKRiEQikYhEIpGIRCKR/uO7IxKZ/Ggukfjg/24kGlkkohXrRPLad+8kEskZ20QikYhEIpFIRCKRSEQikUhEIpFIRCKRSEQikUikBiKRSCQikUgkIpFIJCKRSCT+45u/JRKZmHn07FQkEnnv2UuRyLiQiKzWEon83TdvI5FInG0lEolEJBKJRCISiUQiEolEIhKJRCISiUQiEolEIjUQiUQiEYlEIhGJRCIRiUQiEYlEIhMzEolEIhGJRCIRiUQiEYlEIhGJRCIRiUQikYhEIpGIRCKRiEQikYhEIpGIRCKRSA1EIpFIRCKRSEQikUhEIpFIRCKRyMSMRCKRSEQikUhEIpFIRCKRSEQikUhEIpFIJCKRSCQikUgkIpFIJCKRSCQikUgkUgORSCQSkUgkEpFIJBKRSCQSkUgkMjEjkUgkEpFIJBKRSCQSkUgkEpFIJBKRSCQSiUgkEolIJBKJSCQSiUgkEolIJBKJ1EAkEolEJBKJRCQSiUQkEolEJBKJTMxIJBKJRCQSiUQkEolEJBKJRCQSiUQkEolEIhKJRCISiUQiEolEIhKJRCISiUQiNRCJRCIRiUQiEYlEIhGJRCIRiUQiEzMSiUQiEYlEIhGJRCIRiUQiEYlEIhGJRCKRiEQikYhEIpGIRCKRiEQikYhEIpFIDUQikUhEIpFIRCKRSEQikUhEIpHIxIxEIpFIRCKRSEQikUhEIpFIRCKRSEQikUgkIpFIJCKRSCQikUgkIpFIJCKRSCRSA5FIJBKRSCQSkUgkEpFIJBKRSCQyMSORSCQSkUgkEpFIJBKRSCQSkUgkEpFIJBKJSCQSiUgkEolIJBKJSCQSiUgkEonUQCQSiUQkEolEJBKJRCQSiUQkEon0+8+RSCQSiUgkEolIJBKJSCQSiUgkEolIJBKJRCQSiUQkEolEJBKJRCQSiUQkEolEaiASiUQiEolEIhKJRCISiUQiEolEIhKJRCIRiUQiEYlEIhGJRCIRiUQiEYlEIpGIRCKRiEQikYhEIpGIRCKRiEQikUgNRCKRSEQikUhEIpFIRCKRSEQikUhEIpFIJCKRSCQikUgkIpFIJCKRSCQikUgkEpFIJBKRSCQSkUgkEpFIJBKRSCQSqYFIJBKJSCQSiUgkEolIJBKJSCQSiUgkEolEJBKJRCQSiUQkEolEJBKJRCQSiUQiEolEIhKJRCISiUQiEolEIhKJRCI1EIlEIhGJRCIRiUQiEYlEIhGJRCIRiUQikYhEIpGIRCKRiEQikYhEIpGIRCKRSEQikUhEIpFIRCKRSEQikUhEIpFIpAYikUgkIpFIJCKRSCQikUgkIpFIJCKRSCQSkUgkEpFIJBKRSCQSkUgkEpFIJBKJSCQSiUgkEolIJBKJSCQSiUgkEonUQCQSiUQkEolEJBKJRCQSiUQkEolEJBKJRCISiUQiEolEIhKJRCISiUQiEolEIhGJRCIRiUQiEYlEIhGJRCIRiUQikRqIRCKRiEQikYhEIpGIRCKRiEQikYhEIpFIRCKRSEQikUhEIpFIRCKRSEQikUgkIpFIJCKRSCQikUgkIpFIJCKRSCRSA5FIJBKRSCQSkUgkEpFIJBKRSCQSkUgkEolIJBKJSCQSiUgkEolIJBKJSCQSiUQkEolEJBKJRCQSiUQkEolEJBKJRGogEolEIhKJRCISiUQiEolEIhKJRCISiUQiEYlEIhGJRCIRiUQiEYlEIhGJRCKRiEQikYhEIpGIRCKRiEQikYhE4r8FwQEBAAAAgiB80/+FQSKRGpFIJBKRSCQSkUgkEpFIJBKRSCQSkUgkEolIJBKJSCQSiUgkEolIJBKJSCQSiUQkEolEJBKJRCQSiUQkEolEJBKJRGpEIpFIRCKRSEQikUhEIpFIRCKRSEQikUgkIpFIJCKRSCQikUgkIpFIJCKRSCQSkUgkEpFIJBKRSCQSkUgkEpFIJBKpEYlEIhGJRCIRiUQiEYlEIhGJRCIRiUQikYhEIpGIRCKRiEQikYhEIpGIRCKRSEQikUhEIpFIRCKRSEQikUhEIpFIpEYkEolEJBKJRCQSiUQkEolEJBKJRCQSiUQiEolEIhKJRCISiUQiEolEIhKJRCIRiUQiEYlEIhGJRCIRiUQiEYlEIpEakUgkEpFIJBKRSCQSkUgkEpFIJBKRSCQSiUgkEolIJBKJSCQSiUgkEolIJBKJRCQSiUQkEolEJBKJRCQSiUQkEolEakQikUhEIpFIRCKRSEQikUhEIpFIRCKRSCQikUgkIpFIJCKRSCQikUgkIpFIJBKRSCQSkUgkEpFIJBKRSCQSkUgkEqkRiUQiEYlEIhGJRCIRiUQiEYlEIhGJRCKRiEQikYhEIpGIRCKRiEQikYhEIpFIRCKRSEQikUhEIpFIRCKRSEQikUikRiQSiUQkEolEJBKJRCQSiUQkEolEJBKJRCISiUQiEolEIhKJRCISiUQiEolEIhGJRCIRiUQiEYlEIhGJRCIRiUQikRqRSCQSkUgkEpFIJBKRSCQSkUgkEpFIJBKJSCQSiUgkEolIJBKJSCQSiUgkEolEJBKJRCQSiUQkEolEJBKJRCQSiURqRCKRSEQikUhEIpFIRCKRSEQikUhEIpFIJCKRSCQikUgkIpFIJCKRSCQikUgkEpFIJBKRSCQSkUgkEpFIJBKRSCQSqRGJRCIRiUQiEYlEIhGJRCIRiUQiEYlEIpGIRCKRiEQikYhEIpGIRCKRiEQikUhEIpFIRCKRSEQikUhEIpFIRCKRSKRGJBKJRCQSiUQkEolEJBKJRCQSiUQkEolEIhKJRCISiUQiEolEIhKJRCISiUQiEYlEIhGJRCIRiUQiEYlEIhGJRCKRDlVZyh9BLO8tAAAAAElFTkSuQmCC', 'tuz_biber.png': '...'}
for name, data in EXAMPLES.items():
    (INPUT / name).write_bytes(base64.b64decode(data))
KENDI_GORUNTULERIMI_YUKLE = False
if KENDI_GORUNTULERIMI_YUKLE:
    from google.colab import files
    uploaded = files.upload()
    for name, data in uploaded.items():
        (INPUT / Path(name).name).write_bytes(data)
print('Girdi dizini:', INPUT)
print([p.name for p in INPUT.iterdir()])


## 5. C++ ile 1024×768 boyutlandırma
Üç çalıştırma ayrı dizinlere kaydedilir. CUDA işlemi `cv::cuda::resize` kullanır; Numba kullanılmaz. Tam 1024×768 istendiği için en-boy oranı farklı resimler esnetilir. Çıktı adları `orijinal_ad.uzanti.png` biçimindedir; aynı köklü JPG ve PNG çakışmaz. Var olan çıktılar üzerine yazılmaz.


In [ ]:
import time
import json

timings = {}
for name, options in [('cpu', []), ('parallel', ['--threads', '2']), ('cuda', ['--cuda'])]:
    start = time.perf_counter()
    process = subprocess.run([str(BUILD / 'boyutlandir'), str(INPUT), str(RUN / name), *options],
                             capture_output=True, text=True)
    timings[name] = time.perf_counter() - start
    (RUN / f'{name}.log').write_text(process.stdout + process.stderr, encoding='utf-8')
    print(name, process.stdout, process.stderr)
    process.check_returncode()
(RUN / 'sureler.json').write_text(json.dumps(timings, indent=2), encoding='utf-8')
print('Dosya okuma/yazma ve GPU aktarimi dahil tek calistirma sureleri:', timings)


## 6. Çıktı boyutlarını ve hata durumlarını test et
Çıktı görüntüleri Python ile yalnızca doğrulanır; boyutlandırma C++ programında yapılmıştır. Seri/paralel eşliği, farklı kanal/bit derinlikleri, bozuk dosya, boş dizin ve üzerine yazma koruması da test edilir.

In [ ]:
import cv2
import numpy as np

count = 0
for mode in ('cpu', 'parallel', 'cuda'):
    images = list((RUN / mode).glob('*.png'))
    assert images, f'{mode}: cikti yok'
    for path in images:
        image = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
        assert image is not None and image.shape[:2] == (768, 1024), str(path)
        count += 1
subprocess.run([sys.executable, str(ROOT / 'cpp/test_resize.py'), str(BUILD / 'boyutlandir'),
                '--cuda', '--out', str(RUN / 'testler')], check=True)
print(f'{count} cikti goruntusu 1024x768 olarak dogrulandi.')
from IPython.display import display
from PIL import Image
display(Image.open(next((RUN / 'cuda').glob('*.png'))))


## 7. Sonuçları indir
ZIP; girdi/çıktı görüntülerini, testleri ve CUDA kurulum kanıtını içerir. Ayrıca **Dosya → İndir → .ipynb indir** ile bu defterin çalıştırılmış halini kaydedip GitHub’a yükleyin. Aşağıdaki ZIP bir sonuç arşividir; kaynak kodlar depoda ayrıca bulunmalıdır.

In [ ]:
import shutil
from google.colab import files

for name in ('cuda_dogrulama.txt', 'nvidia-smi.txt', 'nvcc.txt', 'opencv-configure.log',
             'opencv-build.log', 'opencv-install.log'):
    shutil.copy2(ROOT / name, RUN / name)
archive = shutil.make_archive(str(RUN) + '_teslim', 'zip', RUN)
files.download(archive)


## Kaynaklar
- [OpenCV derleme seçenekleri](https://docs.opencv.org/4.12.0/db/d05/tutorial_config_reference.html)
- [OpenCV ek modülleri](https://github.com/opencv/opencv_contrib)
- [OpenCV CUDA](https://docs.opencv.org/4.4.0/d2/dbc/cuda_intro.html)
- [Colab kaynak ve GPU sınırları](https://research.google.com/colaboratory/faq.html)
